[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/04_Buoyancy_Control.ipynb)

# DiveLab

## Notebook 04 — Stabilizing Buoyancy with Feedback Control

**Guiding question:** Can feedback control stabilize a diver whose natural buoyancy dynamics are unstable?

*An unstable plant can become stable when corrective action is fed back from the state.*

## Learning objectives

By the end of this lab, you will be able to:

- distinguish open-loop and closed-loop behavior;
- introduce a control input into the buoyancy model;
- interpret BCD inflation and venting as control actions;
- build a simple linear state-feedback controller;
- compare uncontrolled and controlled trajectories;
- examine closed-loop eigenvalues;
- visualize stabilization in the phase plane.

## From Notebook 03 to Notebook 04

Notebook 03 showed that neutral buoyancy does not imply dynamic stability.

Around the equilibrium:

$$
x_e=
\begin{bmatrix}
z_e\\
0
\end{bmatrix}
$$

small perturbations can grow because of the positive feedback loop:

> displacement → gas-volume change → buoyancy change → more displacement

The linearized equilibrium is a saddle point.

Now we ask:

> Can we deliberately add **negative feedback** to oppose that instability?

## Open loop and closed loop

### Open loop

In the uncontrolled model, the diver evolves according to the natural physics:

$$
\dot x = f(x)
$$

There is no corrective action based on the measured state.

### Closed loop

With feedback control:

$$
\dot x = f(x,u)
$$

and the control input depends on the state:

$$
u = \phi(x)
$$

The controller observes the deviation from the desired equilibrium and acts to reduce it.

## What is the control input?

We introduce a simplified control variable:

$$
u(t)
$$

representing the rate of change of the diver's equivalent surface gas volume.

Interpretation:

- $u>0$: add gas;
- $u<0$: vent gas;
- $u=0$: no control action.

The gas state is therefore dynamic:

$$
\frac{dV_s}{dt}=u
$$

where $V_s$ is the equivalent gas volume referenced to surface pressure.

## Important modeling choice

In reality, BCD inflation and venting involve valve dynamics, gas supply pressure, diver orientation, delays, actuator limits and discrete human actions.

Here we deliberately use a simplified model.

The goal is not to simulate a real BCD valve in detail.

The goal is to understand:

> how feedback can stabilize an unstable dynamical system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Physical constants and equilibrium

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
v_e = 0.0

gas_surface_volume_e = 0.005   # 5 L equivalent surface volume

Cd = 0.9
A_drag = 0.7

In [ ]:
def pressure_at_depth(z):
    return P0 + rho * g * z

def gas_volume_at_depth(z, surface_volume):
    return surface_volume * P0 / pressure_at_depth(z)

## Build neutral buoyancy at the equilibrium

As before, choose the fixed displaced volume so that:

$$
F_B(z_e)=mg
$$

In [ ]:
gas_volume_e = gas_volume_at_depth(z_e, gas_surface_volume_e)
fixed_volume = mass / rho - gas_volume_e

print(f"Equilibrium gas volume at {z_e:.1f} m: {gas_volume_e*1000:.3f} L")
print(f"Fixed displaced volume: {fixed_volume*1000:.3f} L")

In [ ]:
def buoyant_force(z, surface_gas_volume):
    Vg = gas_volume_at_depth(z, surface_gas_volume)
    return rho * g * (fixed_volume + Vg)

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

## Nonlinear state model

We now use three states:

$$
x=
\begin{bmatrix}
z\\
v\\
V_s
\end{bmatrix}
$$

with dynamics:

$$
\dot z=-v
$$

$$
\dot v=
\frac{
F_B(z,V_s)-mg-F_D(v)
}{m}
$$

$$
\dot V_s=u
$$

The third state is important.

In Notebook 03, gas volume was fixed except for Boyle compression and expansion.

Now the controller can actively change the amount of gas in the BCD model.

In [ ]:
def acceleration(z, v, surface_gas_volume):
    return (
        buoyant_force(z, surface_gas_volume)
        - mass * g
        - drag_force(v)
    ) / mass

## First: verify the open-loop instability

We use:

$$
u=0
$$

and apply a small upward velocity disturbance.

In [ ]:
def simulate(controller=None, z0=z_e, v0=0.05, Vs0=gas_surface_volume_e,
             duration=30.0, dt=0.01, u_limit=None):

    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for i in range(n-1):

        if controller is None:
            u = 0.0
        else:
            u = controller(z[i], v[i], Vs[i])

        if u_limit is not None:
            u = np.clip(u, -u_limit, u_limit)

        a = acceleration(z[i], v[i], Vs[i])

        v[i+1] = v[i] + a * dt
        z[i+1] = z[i] - v[i+1] * dt
        Vs[i+1] = max(Vs[i] + u * dt, 0.0)

        u_hist[i] = u

        if z[i+1] <= 0:
            z[i+1:] = 0
            v[i+1:] = v[i+1]
            Vs[i+1:] = Vs[i+1]
            u_hist[i+1:] = u
            break

    u_hist[-1] = u_hist[-2]
    return t, z, v, Vs, u_hist

In [ ]:
t_ol, z_ol, v_ol, Vs_ol, u_ol = simulate(controller=None)

In [ ]:
plt.plot(t_ol, z_ol)
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Open-loop response")
plt.grid(True)
plt.show()

The open-loop diver does not return to $z_e$.

This is the same instability observed in Notebook 03.

Now we introduce feedback.

## A first feedback idea

Suppose we define the errors:

$$
e_z=z-z_e
$$

and:

$$
e_v=v-v_e=v
$$

A simple state-feedback law is:

$$
u = K_z e_z - K_v e_v
$$

Interpretation:

- if the diver is too deep ($e_z>0$), add gas;
- if the diver is too shallow ($e_z<0$), vent gas;
- if upward velocity is positive, vent;
- if downward velocity is negative, add gas.

This is negative feedback.

## Why the signs matter

Recall our conventions:

- depth increases downward;
- velocity is positive upward.

So if:

$$
z>z_e
$$

the diver is too deep and needs more buoyancy, therefore $u>0$.

If:

$$
v>0
$$

the diver is already moving upward, so the controller should reduce buoyancy, therefore the velocity-feedback contribution must be negative.

In [ ]:
Kz = 0.00008
Kv = 0.0008

def state_feedback_controller(z, v, Vs):
    e_z = z - z_e
    e_v = v
    return Kz * e_z - Kv * e_v

## Closed-loop simulation

In [ ]:
t_cl, z_cl, v_cl, Vs_cl, u_cl = simulate(
    controller=state_feedback_controller,
    z0=z_e,
    v0=0.05,
    duration=30.0,
    dt=0.01,
    u_limit=0.0005
)

In [ ]:
plt.plot(t_ol, z_ol, label="Open loop")
plt.plot(t_cl, z_cl, label="Closed loop")
plt.axhline(z_e, linestyle="--", label="Target depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Open-loop vs closed-loop depth response")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t_ol, v_ol, label="Open loop")
plt.plot(t_cl, v_cl, label="Closed loop")
plt.axhline(0, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Open-loop vs closed-loop velocity")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

The controller uses the state itself to oppose the natural instability.

The loop becomes:

> depth / velocity error → controller → gas change → buoyancy correction → reduced error

This is a **negative feedback loop**.

Compare it with the natural positive-feedback loop:

> displacement → gas expansion/compression → buoyancy change → more displacement

## What does the controller actually do?

In [ ]:
plt.plot(t_cl, u_cl)

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("BCD control action")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(t_cl, Vs_cl * 1000)

plt.xlabel("Time [s]")
plt.ylabel("Equivalent surface gas volume [L]")
plt.title("Controlled gas-volume state")
plt.grid(True)
plt.show()

The controller does not directly command depth.

It changes gas volume.

Gas volume changes buoyancy.

Buoyancy changes acceleration.

Acceleration changes velocity.

Velocity changes depth.

So the actuator affects depth through the plant dynamics.

## Linear control model

Near equilibrium, define perturbations:

$$
\delta z=z-z_e
$$

$$
\delta v=v
$$

$$
\delta V_s=V_s-V_{s,e}
$$

The linearized system has the form:

$$
\delta\dot x=A\delta x+B\delta u
$$

## Compute the local coefficients

We already know:

$$
\frac{\partial F_B}{\partial z}<0
$$

We also need the effect of gas input on buoyancy:

$$
\frac{\partial F_B}{\partial V_s}
=
\rho g \frac{P_0}{P(z_e)}
$$

In [ ]:
dFb_dz = (
    -rho * g
    * gas_surface_volume_e
    * P0
    * rho * g
    / pressure_at_depth(z_e)**2
)

dFb_dVs = rho * g * P0 / pressure_at_depth(z_e)

print(f"dFb/dz  = {dFb_dz:.3f} N/m")
print(f"dFb/dVs = {dFb_dVs:.1f} N/m^3")

Ignoring quadratic drag at first order around $v=0$, the linearized model is:

$$
\begin{bmatrix}
\delta\dot z\\
\delta\dot v\\
\delta\dot V_s
\end{bmatrix}
=
\begin{bmatrix}
0 & -1 & 0\\
a_z & 0 & a_V\\
0 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
\delta z\\
\delta v\\
\delta V_s
\end{bmatrix}
+
\begin{bmatrix}
0\\
0\\
1
\end{bmatrix}
u
$$

where:

$$
a_z=\frac{1}{m}\frac{\partial F_B}{\partial z}
$$

and:

$$
a_V=\frac{1}{m}\frac{\partial F_B}{\partial V_s}
$$

In [ ]:
a_z = dFb_dz / mass
a_V = dFb_dVs / mass

A_lin = np.array([
    [0.0, -1.0, 0.0],
    [a_z, 0.0, a_V],
    [0.0, 0.0, 0.0]
])

B_lin = np.array([
    [0.0],
    [0.0],
    [1.0]
])

print("A =")
print(A_lin)
print()
print("B =")
print(B_lin)

## Closed-loop state-space form

Our controller can be written:

$$
u=-K\delta x
$$

with a suitable gain vector $K$.

Then:

$$
\delta\dot x=(A-BK)\delta x
$$

The matrix:

$$
A_{cl}=A-BK
$$

determines local closed-loop stability.

For our sign convention:

$$
u=K_z\delta z-K_v\delta v
$$

and we do not yet directly feed back $\delta V_s$.

So:

$$
K=
\begin{bmatrix}
-K_z & K_v & 0
\end{bmatrix}
$$

In [ ]:
K = np.array([[-Kz, Kv, 0.0]])

A_cl = A_lin - B_lin @ K

eig_open = np.linalg.eigvals(A_lin)
eig_closed = np.linalg.eigvals(A_cl)

print("Open-loop eigenvalues:")
print(eig_open)
print()
print("Closed-loop eigenvalues:")
print(eig_closed)

## A subtle but important result

With only depth and velocity feedback acting through the **integral gas state** $V_s$, the closed-loop structure may not behave exactly like a simple second-order PD controller.

The actuator itself adds a state:

$$
\dot V_s=u
$$

This means controller design must account for the actuator dynamics.

This is a useful systems-engineering lesson:

> adding an actuator can change the order of the system.

## Phase-plane comparison

Although the complete controlled model has three states, we can still project trajectories onto the $(z,v)$ plane.

In [ ]:
plt.plot(z_ol, v_ol, label="Open loop")
plt.plot(z_cl, v_cl, label="Closed loop")
plt.scatter([z_e], [0], s=70, label="Equilibrium")

plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Projected phase portrait: open vs closed loop")
plt.grid(True)
plt.legend()
plt.show()

## What stabilization means

For the closed-loop system, we want:

$$
z(t)\rightarrow z_e
$$

$$
v(t)\rightarrow0
$$

and ideally:

$$
V_s(t)\rightarrow V_{s,e}
$$

after small disturbances.

In state-space language:

$$
x(t)\rightarrow x_e
$$

## Control effort and actuator limits

Real actuators cannot produce unlimited control action.

That is why the simulation includes:

```python
u_limit
```

This represents actuator saturation.

Saturation matters because a controller that is mathematically stabilizing without limits may behave very differently when its commanded action exceeds what the actuator can physically deliver.

## Human diver as controller

A human diver performs a feedback-control task:

1. senses depth, velocity and buoyancy tendency;
2. compares them with the desired state;
3. decides whether corrective action is needed;
4. adds or vents gas;
5. observes the resulting response;
6. repeats.

The diver is therefore part of the closed loop.

## Delay

Human control is not instantaneous.

There is:

- sensing delay;
- decision delay;
- actuation delay;
- plant response delay.

Delay is important because excessive delay in an unstable system can make control much harder.

We will model delay explicitly in a later notebook.

## Exercises

### 1. Change the gains

Try different values of:

```python
Kz
Kv
```

Observe:

- settling time;
- overshoot;
- control effort;
- oscillation.

### 2. Remove velocity feedback

Set:

```python
Kv = 0
```

Does depth feedback alone behave well?

Why might velocity feedback be useful?

### 3. Change actuator limits

Try:

```python
u_limit = 0.0001
u_limit = 0.0005
u_limit = 0.001
```

How does saturation affect the response?

### 4. Opposite perturbation

Start with:

```python
v0 = -0.05
```

Can the same controller recover from a downward disturbance?

## Challenge — pole placement intuition

The open-loop system has unstable dynamics.

The goal of feedback design is to move the closed-loop eigenvalues to locations associated with stable behavior.

Explore the eigenvalues of:

$$
A-BK
$$

for different gain choices.

Can you find gains for which all closed-loop eigenvalues have negative real part?

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- the natural buoyancy system can be open-loop unstable;
- BCD gas change can be modeled as a control input;
- state feedback can oppose the positive buoyancy loop;
- depth and velocity errors can be used to generate corrective action;
- actuator dynamics increase the order of the system;
- actuator saturation matters;
- closed-loop eigenvalues provide a local stability test;
- the diver can be interpreted as a feedback controller.

### Core control loop

> state error → corrective gas action → buoyancy change → motion correction → reduced state error

### Next

Notebook 05 can study a crucial real-world complication:

> **What happens when feedback is delayed?**

That will let us explore oscillation, overcorrection and loss of stability.